6k image --
784 feature----> 128 layuar-->64 neuron layar--->10 neuron---->ypred among 10 class

In [2]:
import torch
from torch.utils.data import Dataset,DataLoader
import pandas as pd
import numpy as np
import torch.nn as nn
from sklearn.model_selection import train_test_split
import torch.optim as optim

In [3]:
torch.manual_seed(42)

In [5]:
df=pd.read_csv('fmnist_small.csv')
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,9,0,0,0,0,0,0,0,0,0,...,0,7,0,50,205,196,213,165,0,0
1,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,1,0,0,0,...,142,142,142,21,0,3,0,0,0,0
3,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,8,0,0,0,0,0,0,0,0,0,...,213,203,174,151,188,10,0,0,0,0


In [6]:
x=df.iloc[:,1:-1]
y=df.iloc[:,0]
x.shape

(6000, 783)

In [7]:
xtrain,xtest,ytrain,ytest=train_test_split(x,y,test_size=0.2)

# scaling the train dataset (255 is max pixelsize)
xtrain=xtrain/255.0
xtest=xtest/255.0

In [8]:
# creating a costomDataset class

class CostomDataset(Dataset):
  def __init__(self,feature,labels):
    self.feature= torch.tensor(feature.values,dtype=torch.float32)
    self.labels=torch.tensor(labels.values, dtype=torch.long)

  def __len__(self):
    return  len(self.feature)

  def __getitem__(self, index):
    return self.feature[index],self.labels[index]

Passing a Pandas DataFrame directly to torch.tensor()
In your __init__ method, you are trying to convert feature directly into a tensor using torch.tensor(feature). Because xtrain is a Pandas DataFrame (from xtrain=df.iloc[:, 1:-1]), PyTorch doesn't know how to interpret its shape directly, resulting in:

ValueError: could not determine the shape of object type 'DataFrame'

The fix: Convert your DataFrame (or Series for labels) into a NumPy array first using .values before wrapping it in torch.tensor()

Note: Also remember that Dataset objects in PyTorch use len(train_dataset) instead of .shape, so in Cell 21 you should check len(train_dataset) instead of train_dataset.shape).




In [9]:
# creating train dataset object
train_dataset=CostomDataset(xtrain,ytrain)
len(train_dataset)
# train_dataset[0]

4800

In [10]:
test_dataset=CostomDataset(xtest,ytest)
len(test_dataset)

1200

In [11]:
# train and test loader
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=32,shuffle=False)

# during predition we off the shufling in we cant get acuracy when
#  data is geting change

In [12]:
from torch.nn.modules.linear import Linear

# define NN class
class MyNN(nn.Module):
  def __init__(self,num_feature):
     super().__init__()
     self.Model=nn.Sequential(
          nn.Linear(num_feature,128),
          nn.ReLU(),
          nn.Linear(128,64),
          nn.ReLU(),
          nn.Linear(64,10)
      )
  def forward(self,x):
    return self.Model(x)


In [13]:
# setting learning rate and epoch

epochs=100
learning_rate=0.1

In [14]:
#initiating the model
model=MyNN(xtrain.shape[1])

# loss functin

criterion=nn.CrossEntropyLoss()

# optimizer
optimizer=optim.SGD(model.parameters(),lr=learning_rate)


In [15]:
# training loop
for epoch in range(epochs):
  total_epoch_loss = 0
  for batch_feature,batch_labels in train_loader:
    # forward pass
    output=model(batch_feature)

    # calculate
    loss=criterion(output,batch_labels)

    # backpass
    optimizer.zero_grad()
    loss.backward()

    # update grads
    optimizer.step()

    total_epoch_loss=total_epoch_loss + loss.item()
  avg_loss =total_epoch_loss/len(train_loader)
  print(f'Epoch :{epoch+1} ,Loss : {avg_loss}')


Epoch :1 ,Loss : 1.3162846342722574
Epoch :2 ,Loss : 0.7631642196575801
Epoch :3 ,Loss : 0.6427275935808817
Epoch :4 ,Loss : 0.5857980179786683
Epoch :5 ,Loss : 0.5408931442101796
Epoch :6 ,Loss : 0.5104929119348526
Epoch :7 ,Loss : 0.47188551266988116
Epoch :8 ,Loss : 0.44820426175991696
Epoch :9 ,Loss : 0.42037530466914175
Epoch :10 ,Loss : 0.4052977919578552
Epoch :11 ,Loss : 0.38322644526759786
Epoch :12 ,Loss : 0.3705923185745875
Epoch :13 ,Loss : 0.3600130006670952
Epoch :14 ,Loss : 0.3389241455992063
Epoch :15 ,Loss : 0.3260303202271462
Epoch :16 ,Loss : 0.31401156763235727
Epoch :17 ,Loss : 0.2983101891477903
Epoch :18 ,Loss : 0.2945906636118889
Epoch :19 ,Loss : 0.281207425793012
Epoch :20 ,Loss : 0.2675099605321884
Epoch :21 ,Loss : 0.2636359831690788
Epoch :22 ,Loss : 0.2630219742655754
Epoch :23 ,Loss : 0.24902703799307346
Epoch :24 ,Loss : 0.24676863946020602
Epoch :25 ,Loss : 0.2237858985364437
Epoch :26 ,Loss : 0.22798035442829132
Epoch :27 ,Loss : 0.2160566940655311
Epo

In [16]:
model.eval()

MyNN(
  (Model): Sequential(
    (0): Linear(in_features=783, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [17]:
total=0
corect=0
with torch.no_grad():

  for batch_feature,batch_labels in test_loader:
    output=model(batch_feature)
    _, predicted= torch.max(output,1)
    total=total+batch_labels.shape[0]
    corect = corect+(predicted == batch_labels).sum().item()
print(corect/total)


0.855


# USING GPU

In [18]:
import torch
import torch.optim as optim
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [29]:
torch.manual_seed(42)

In [31]:
train_df=pd.read_csv("fashion-mnist_train.csv")
train_df.shape

(60000, 785)

In [32]:
test_df=pd.read_csv("fashion-mnist_test.csv")
test_df.shape

(10000, 785)

In [33]:
xtrain=train_df.iloc[:,1:]/255
ytrain=train_df.iloc[:,0]
# ytrain.head()

In [34]:
xtest=test_df.iloc[:,1:]/255
ytest=test_df.iloc[:,0]
ytest.shape

(10000,)

In [35]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"using device: {device}")

using device: cuda


In [36]:
class CustomDataset(Dataset):
  def __init__(self,feature,labels):
    # convrting to pytorch tensor
    self.features= torch.tensor(feature.values,dtype=torch.float32)
    self.labels=torch.tensor(labels.values,dtype=torch.long)

  def __len__(self):
    return len(self.features)

  def __getitem__(self,index):
    return self.features[index],self.labels[index]


In [37]:

train_dataset=CustomDataset(xtrain,ytrain)
len(train_dataset)
test_dataset =CustomDataset(xtest,ytest)
len(test_dataset)

10000

In [38]:
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader=DataLoader(test_dataset,batch_size=32,shuffle=False)

In [39]:
len(train_loader) #60k/32 = 1875

1875

In [40]:
class MyNN(nn.Module):
  def __init__(self,num_feature):
    super().__init__()
    self.model = nn.Sequential(
        nn.Linear(num_feature,128),
        nn.ReLU(),
        nn.Linear(128,64),
        nn.ReLU(),
        nn.Linear(64,10)
    )
  def forward(self,x):
      return self.model(x)

In [41]:
learning_rate=0.1
epoches =100

In [42]:

# creating model
model=MyNN(xtrain.shape[1])

model=model.to(device)

loss_fun= nn.CrossEntropyLoss()

optimizer=optim.SGD(model.parameters(),lr=learning_rate)

In [43]:
# training loop

for epoch in range(epoches):
  total_epoch_loss=0
  for batch_feature,batch_label in train_loader:

    batch_feature,batch_label = batch_feature.to(device),batch_label.to(device)

    # forward pass
    output=model(batch_feature)

    # losscalculation
    loss= loss_fun(output,batch_label)

    # backpropagation
    optimizer.zero_grad()
    loss.backward()

    # update
    optimizer.step()

    total_epoch_loss+=loss.item()

  avg_loss = total_epoch_loss/len(train_loader)
  print(f'Epoch :{epoch+1}, Loss :{avg_loss}')


Epoch :1, Loss :0.6010189415454864
Epoch :2, Loss :0.41350762896935145
Epoch :3, Loss :0.3712131387670835
Epoch :4, Loss :0.34375916006366414
Epoch :5, Loss :0.32385403326948486
Epoch :6, Loss :0.30911719589829445
Epoch :7, Loss :0.2962389792541663
Epoch :8, Loss :0.28478201231459777
Epoch :9, Loss :0.2734388600885868
Epoch :10, Loss :0.2657709174454212
Epoch :11, Loss :0.2583073012312253
Epoch :12, Loss :0.2509995386739572
Epoch :13, Loss :0.24375855010946593
Epoch :14, Loss :0.23399952282508216
Epoch :15, Loss :0.23218019322057565
Epoch :16, Loss :0.22519358284076055
Epoch :17, Loss :0.22086198432445525
Epoch :18, Loss :0.2151347258021434
Epoch :19, Loss :0.2107167779251933
Epoch :20, Loss :0.20593826397756734
Epoch :21, Loss :0.201329383512338
Epoch :22, Loss :0.1974795329729716
Epoch :23, Loss :0.19477506672739983
Epoch :24, Loss :0.18993150207748016
Epoch :25, Loss :0.18798903930981953
Epoch :26, Loss :0.18390919292271138
Epoch :27, Loss :0.1810613950351874
Epoch :28, Loss :0.1760

In [44]:
model.eval()

MyNN(
  (model): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [45]:
total=0
corect=0
with torch.no_grad():
  for batch_feature,batch_label in test_loader:
    batch_feature,batch_label= batch_feature.to(device),batch_label.to(device)
    output=model(batch_feature)
    _ ,predicted=torch.max(output,1)
    total+=batch_label.shape[0]
    corect+=(predicted==batch_label).sum().item()
print(corect/total)

0.8811
